In [ ]:
import pretty_midi
import os
import shutil
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import glob

import pretty_midi
import numpy as np

def align_midi_to_beats(input_midi_path: str, output_midi_path: str):
    """
    Quantizes a MIDI file to the nearest 16th note, accounting for tempo changes.

    Args:
        input_midi_path (str): Path to the source MIDI file.
        output_midi_path (str): Path to save the aligned MIDI file.
    """
    # 1. Load the MIDI file
    midi_data = pretty_midi.PrettyMIDI(input_midi_path)
    
    # 2. Get the beat times, which serve as our main grid
    beat_times = midi_data.get_beats()
    
    # 3. Define the quantization grid (4 subdivisions per beat = 16th notes)
    subdivisions_per_beat = 4
    
    # 4. Create a master list of all 16th-note grid times in seconds
    grid_times = []
    for i in range(len(beat_times) - 1):
        # Get the start and end time of the current beat
        beat_start = beat_times[i]
        beat_end = beat_times[i+1]
        beat_duration = beat_end - beat_start
        
        # Calculate the time for each subdivision within this beat
        subdivision_duration = beat_duration / subdivisions_per_beat
        for j in range(subdivisions_per_beat):
            grid_times.append(beat_start + j * subdivision_duration)
    
    # Add the final beat time to complete the grid
    grid_times.append(beat_times[-1])
    grid_times = np.array(grid_times) # Convert to numpy array for faster search

    print(f"Created a quantization grid with {len(grid_times)} points.")

    # 5. Snap each note to the nearest grid time
    for instrument in midi_data.instruments:
        for note in instrument.notes:
            # Find the index of the closest grid time to the note's start
            closest_grid_index = np.argmin(np.abs(grid_times - note.start))
            
            # Get the new, quantized start time
            new_start_time = grid_times[closest_grid_index]
            
            # Preserve the note's original duration
            original_duration = note.end - note.start
            
            # Update the note's timing
            note.start = new_start_time
            note.end = new_start_time + original_duration
            
    # 6. Save the new, aligned MIDI file
    midi_data.write(output_midi_path)
    print(f"Successfully saved aligned MIDI to '{output_midi_path}'")

# --- Example Usage ---
# align_midi_to_beats('my_unaligned_song.mid', 'my_aligned_song.mid')

# --- Configuration ---
# The suspicious ratio range: 0.5 ± 0.15
SUSPICIOUS_RANGE = (0.35, 0.65)

def filter_midi_file(midi_path: str) -> tuple[bool, float]:
    """
    Applies the rule-based filter to a MIDI file to check for poor quantization.

    The rule discards a song if the ratio of notes on odd vs. even time steps
    falls within the SUSPICIOUS_RANGE for EVERY track in the song.

    Args:
        midi_path (str): The path to the MIDI file.

    Returns:
        bool: True if the file should be KEPT, False if it should be DISCARDED.
    """
    try:
        midi_data = pretty_midi.PrettyMIDI(midi_path)
    except Exception as e:
        print(f"  - WARNING: Could not parse {Path(midi_path).name}. Discarding. Error: {e}")
        return False, 0.5 # Discard unparseable files

    if not midi_data.instruments:
        return False, 0.5 # Discard empty MIDI files with no tracks

    # --- The Core Logic ---
    # We assume a file is "bad" (all tracks are suspicious) until we find proof
    # to the contrary (i.e., at least one "good" track).
    all_tracks_are_suspicious = True
    # print('There are instruments')
    time_steps = []
    for instrument in midi_data.instruments:
        # A track with no notes cannot be suspicious, so it breaks the "every track" condition.
        if not instrument.notes:
            all_tracks_are_suspicious = False
            break

        odd_ticks = 0
        even_ticks = 0

        for note in instrument.notes:            # Convert note's start time from seconds to an integer "time step" (tick)
            time_step = midi_data.time_to_tick(note.start) // 10
            time_steps.append(time_step)
            if time_step % 2 == 0:
                even_ticks += 1
            else:
                odd_ticks += 1

        # Avoid division by zero. A track with no even notes has an infinite ratio,
        # which is well outside the suspicious range, so it's a "good" track.
        if even_ticks == 0:
            is_suspicious = False
        else:
            # ratio = odd_ticks / (odd_ticks + even_ticks)
            ratio = odd_ticks / even_ticks
            # Check if the ratio falls within the "suspicious" range
            if SUSPICIOUS_RANGE[0] <= ratio <= SUSPICIOUS_RANGE[1]:
                is_suspicious = True
            else:
                is_suspicious = False

        # If we find even one track that is NOT suspicious, the whole file is good.
        if not is_suspicious:
            all_tracks_are_suspicious = False
            break # No need to check other tracks

    # A file is discarded ONLY IF all of its tracks were suspicious.
    # Therefore, we KEEP the file if `all_tracks_are_suspicious` is False.
    return not all_tracks_are_suspicious, ratio#, time_steps


def process_directory(source_dir: str, keep_dir: str, discard_dir: str):
    """
    Processes a directory of MIDI files, sorting them into subfolders.
    """
    source_path = Path(source_dir)
    if not source_path.is_dir():
        print(f"Error: Source directory '{source_dir}' not found.")
        return

    # Path(keep_dir).mkdir(exist_ok=True)
    # Path(discard_dir).mkdir(exist_ok=True)

    print(f"Scanning MIDI files in '{source_dir}'...")
    midi_files = list(source_path.glob('**/*.mid', recurse_symlinks=True))#[:100]

    if not midi_files:
        print("No MIDI files found in the source directory.")
        return

    kept_count, discarded_count = 0, 0
    ratio = 0
    ratios = []
    pbar = tqdm(midi_files, desc="Processing MIDI files", total=len(midi_files), dynamic_ncols=True)
    pbar.set_postfix(discarded=0, kept=0, ratio=0, refresh=True)

    # Use tqdm to show progress bar while tracking the number of discarded files
    for i, midi_file in enumerate(pbar):
        should_keep, ratio = filter_midi_file(str(midi_file))
        ratios.append(ratio)
        # if i % 100 == 0:
        #     print(f"Ratio: {np.max(ratios)}")
        if should_keep:
            # print(f"-> Keeping: {midi_file.name}")
            if keep_dir:
                shutil.copy(midi_file, Path(keep_dir) / midi_file.name)
            kept_count += 1
        else:
            # print(f"-> Discarding: {midi_file.name}")
            if discard_dir:
                shutil.copy(midi_file, Path(discard_dir) / midi_file.name)
            discarded_count += 1
        pbar.set_postfix(discarded=discarded_count, kept=kept_count, ratio=ratio, refresh=True)
    
    print("\n--- Filtering Complete ---")
    print(f"✅ Files Kept: {kept_count}")
    print(f"❌ Files Discarded: {discarded_count}")
    print(f"Results moved to '{keep_dir}' and '{discard_dir}'.")
    print(ratios)
    return ratios

In [ ]:
# data_path = '../aria-midi-v1-unique-ext/data'
# ratios = process_directory(data_path, 'data_ratio_direct/keep', 'data_ratio_direct/discard')

Scanning MIDI files in '../aria-midi-v1-unique-ext/data'...


Processing MIDI files: 100%|██████████| 32522/32522 [15:12<00:00, 35.65it/s, discarded=57, kept=32465, ratio=1.03]  


--- Filtering Complete ---
✅ Files Kept: 32465
❌ Files Discarded: 57
Results moved to 'data_ratio_direct/keep' and 'data_ratio_direct/discard'.
[0.9474747474747475, 0.9956312800349497, 0.978125, 0.9195402298850575, 0.9724137931034482, 0.935064935064935, 0.972663139329806, 1.0670731707317074, 1.0124610591900312, 1.0364077669902914, 1.0303951367781155, 0.9847715736040609, 0.9223744292237442, 0.9147640791476408, 1.0278179190751444, 1.0353697749196142, 0.9299835255354201, 0.8917910447761194, 0.9564459930313589, 1.0400943396226414, 0.8883720930232558, 0.9810495626822158, 0.9712313003452244, 0.9130434782608695, 0.9293544457978076, 0.9930167597765364, 0.9272197962154294, 0.9444444444444444, 1.0173697270471465, 1.098550724637681, 0.9375, 1.0638297872340425, 1.0814814814814815, 1.0028818443804035, 0.8376068376068376, 1.0, 0.9870410367170627, 1.2454545454545454, 1.0576131687242798, 1.0, 1.0267558528428093, 0.9148514851485149, 0.9349112426035503, 0.8473520249221184, 1.0236220472440944, 0.9273153

In [13]:
np.histogram(np.abs(ratios), bins=100)

(array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 100,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0]),
 array([-0.5 , -0.49, -0.48, -0.47, -0.46, -0.45, -0.44, -0.43, -0.42,
        -0.41, -0.4 , -0.39, -0.38, -0.37, -0.36, -0.35, -0.34, -0.33,
        -0.32, -0.31, -0.3 , -0.29, -0.28, -0.27, -0.26, -0.25, -0.24,
        -0.23, -0.22, -0.21, -0.2 , -0.19, -0.18, -0.17, -0.16, -0.15,
        -0.14, -0.13, -0.12, -0.11, -0.1 , -0.09, -0.08, -0.07, -0.06,
        -0.05, -0.04, -0.03, -0.02, -0.01,  0.  ,  0.01,  0.02,  0.03,
        

In [78]:
path = '../aria-midi-v1-unique-ext/data/aa/000173_0.mid'
# filter_midi_file(path)
align_midi_to_beats(path, 'test/' + Path(path).name)

Created a quantization grid with 901 points.
Successfully saved aligned MIDI to 'test/000173_0.mid'


In [39]:
import tempfile

# --- Simple Test Function ---
def run_filter_tests():
    """
    Creates temporary MIDI files to test the 'should_keep_midi' function.
    """
    print("--- Running MIDI Filter Tests ---")

    def _create_test_midi(note_placements: dict, filename: str) -> str:
        """A helper function to generate a MIDI file for testing."""
        midi = pretty_midi.PrettyMIDI()
        for i, (track_name, ticks) in enumerate(note_placements.items()):
            instrument = pretty_midi.Instrument(program=0, name=track_name)
            for tick in ticks:
                time_s = midi.tick_to_time(tick)
                note = pretty_midi.Note(
                    velocity=100, pitch=60, start=time_s, end=time_s + 0.1
                )
                instrument.notes.append(note)
            midi.instruments.append(instrument)
        
        filepath = os.path.join(tempfile.gettempdir(), filename)
        midi.write(filepath)
        return filepath

    # Test Case 1: Create a file that SHOULD BE DISCARDED
    print("\nRunning Test Case 1: Should Discard")
    discard_notes = {
        'Piano':   [1, 3, 2, 4, 6, 8],  # Ratio 2/4 = 0.5 (Suspicious)
        'Strings': [5, 10, 12]           # Ratio 1/2 = 0.5 (Suspicious)
    }
    discard_path = _create_test_midi(discard_notes, "discard_me.mid")
    result_discard, ratio = filter_midi_file(discard_path)
    print(f"Ratio: {ratio}")
    
    if not result_discard:
        print("-> ✅ PASSED: File with suspicious ratios was correctly discarded.")
    else:
        print("-> ❌ FAILED: File with suspicious ratios was incorrectly kept.")
    
    os.remove(discard_path) # Clean up the temporary file

    # Test Case 2: Create a file that SHOULD BE KEPT
    print("\nRunning Test Case 2: Should Keep")
    keep_notes = {
        'Drums': [1, 3, 5, 2, 4, 6],  # Ratio 3/3 = 1.0 (Not suspicious)
        'Bass':  [7, 9, 8, 10]        # Ratio 2/2 = 1.0 (Not suspicious)
    }
    keep_path = _create_test_midi(keep_notes, "keep_me.mid")
    result_keep = filter_midi_file(keep_path)

    if result_keep:
        print("-> ✅ PASSED: File with a 1:1 ratio was correctly kept.")
    else:
        print("-> ❌ FAILED: File with a 1:1 ratio was incorrectly discarded.")

    os.remove(keep_path) # Clean up the temporary file
    print("\n--- Tests Complete ---")


run_filter_tests()

--- Running MIDI Filter Tests ---

Running Test Case 1: Should Discard
Ratio: 0.0
-> ❌ FAILED: File with suspicious ratios was incorrectly kept.

Running Test Case 2: Should Keep
-> ✅ PASSED: File with a 1:1 ratio was correctly kept.

--- Tests Complete ---
